In [1]:
import cv2
import random
import numpy as np
import albumentations as A
from pathlib import Path
from tqdm import tqdm

# ── CONFIG ──────────────────────────────────────────────────────────────────
INPUT_DIR   = r"../data/raw_tin"
OUTPUT_DIR  = r"../data/augmented_tin"
MULTIPLIER  = 8
TARGET_SIZE = 640
SEED        = 42
MIN_AREA_PX = 100 # minimum pixel area for a surviving mask instance
# ────────────────────────────────────────────────────────────────────────────

random.seed(SEED)
np.random.seed(SEED)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# ── AUGMENTATION PIPELINE ───────────────────────────────────────────────────
transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),

        A.Rotate(
            limit=180,
            border_mode=cv2.BORDER_CONSTANT,
            fill=0, # image fill for rotated corners
            fill_mask=0, # mask fill for rotated corners
            p=0.8,
        ),

        A.RandomScale(
            scale_limit=(-0.30, 0.0),
            p=0.5,
        ),

        A.RandomResizedCrop(
            size=(TARGET_SIZE, TARGET_SIZE),
            scale=(0.80, 1.0),
            ratio=(1.0, 1.0),
            p=0.5,
        ),

        A.PadIfNeeded(
            min_height=TARGET_SIZE,
            min_width=TARGET_SIZE,
            border_mode=cv2.BORDER_CONSTANT,
            fill=0,
            fill_mask=0,
        ),

        A.Resize(height=TARGET_SIZE, width=TARGET_SIZE),

        # Colour augmentations — do not affect mask
        A.RandomBrightnessContrast(
            brightness_limit=(-0.40, 0.15),
            contrast_limit=0,
            p=0.7,
        ),

        A.RandomGamma(
            gamma_limit=(70, 130),
            p=0.5,
        ),

        A.HueSaturationValue(
            hue_shift_limit=25,
            sat_shift_limit=30,
            val_shift_limit=0,
            p=0.7,
        ),

        A.GaussianBlur(
            blur_limit=(3, 5),
            p=0.4,
        ),

        A.GaussNoise(
            std_range=(0.005, 0.01),
            p=0.4,
        ),
    ],
    bbox_params=A.BboxParams(
        format="yolo",
        label_fields=["class_labels"],
        min_visibility=0.3,
        clip=True,
    ),
)


# ── HELPERS ─────────────────────────────────────────────────────────────────

def read_yolo_label(label_path):
    if not label_path.exists():
        return []
    lines = label_path.read_text().strip().splitlines()
    entries = []
    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue
        cls = int(parts[0])
        coords = list(map(float, parts[1:]))
        entries.append((cls, coords))
    return entries


def is_segmentation(entries):
    for _, coords in entries:
        if len(coords) > 4:
            return True
    return False


def seg_to_bboxes(entries):
    bboxes = []
    class_labels = []
    for cls, coords in entries:
        xs = coords[0::2]
        ys = coords[1::2]
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        cx = (x_min + x_max) / 2
        cy = (y_min + y_max) / 2
        w  = x_max - x_min
        h  = y_max - y_min
        bboxes.append([cx, cy, w, h])
        class_labels.append(cls)
    return bboxes, class_labels


def polygons_to_mask(entries, img_w, img_h):
    """
    Rasterise YOLO segmentation polygons into a uint8 instance mask.
    Each object gets a unique pixel value (1, 2, 3...).
    Value 0 = background.
    Supports up to 254 instances per image.
    """
    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    class_map = {}

    for instance_id, (cls, coords) in enumerate(entries, start=1):
        if instance_id > 254:
            tqdm.write(f"  [WARN] More than 254 instances in one image, skipping extras")
            break
        xs = coords[0::2]
        ys = coords[1::2]
        pts = np.array(
            [[int(x * img_w), int(y * img_h)] for x, y in zip(xs, ys)],
            dtype=np.int32,
        )
        cv2.fillPoly(mask, [pts], color=instance_id)
        class_map[instance_id] = cls

    return mask, class_map


def mask_to_polygons(mask, class_map, img_w, img_h, min_area_px=100):
    """
    Extract YOLO segmentation polygons from a transformed instance mask.
    Skips instances that were fully cropped or rotated out of frame,
    or whose surviving area is below min_area_px.
    """
    entries = []
    for instance_id, cls in class_map.items():
        binary = (mask == instance_id).astype(np.uint8)
        contours, _ = cv2.findContours(
            binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        if not contours:
            continue

        contour = max(contours, key=cv2.contourArea)
        if cv2.contourArea(contour) < min_area_px:
            continue

        epsilon = 0.002 * cv2.arcLength(contour, True)
        contour = cv2.approxPolyDP(contour, epsilon, True)

        if len(contour) < 3:
            continue

        coords = []
        for pt in contour:
            x_norm = max(0.0, min(1.0, float(pt[0][0]) / img_w))
            y_norm = max(0.0, min(1.0, float(pt[0][1]) / img_h))
            coords.extend([x_norm, y_norm])

        entries.append((cls, coords))

    return entries


def write_yolo_label(label_path, entries):
    lines = []
    for cls, coords in entries:
        coord_str = " ".join(f"{v:.6f}" for v in coords)
        lines.append(f"{cls} {coord_str}")
    label_path.write_text("\n".join(lines))


# ── SANITY CHECK ─────────────────────────────────────────────────────────────
in_img_dir  = Path(INPUT_DIR)  / "train" / "images"
in_lbl_dir  = Path(INPUT_DIR)  / "train" / "labels"
out_img_dir = Path(OUTPUT_DIR) / "train" / "images"
out_lbl_dir = Path(OUTPUT_DIR) / "train" / "labels"

assert in_img_dir.exists(), f"Images folder not found: {in_img_dir}"
assert in_lbl_dir.exists(), f"Labels folder not found: {in_lbl_dir}"

out_img_dir.mkdir(parents=True, exist_ok=True)
existing = list(out_img_dir.glob("*"))
if existing:
    print(f"  WARNING: Output folder already has {len(existing)} files.")
    print(f"  Existing files will be overwritten.")
    print(f"  Press Ctrl+C to cancel, or wait 5 seconds to continue...")
    import time
    time.sleep(5)
out_lbl_dir.mkdir(parents=True, exist_ok=True)

img_files = sorted([f for f in in_img_dir.iterdir() if f.suffix.lower() in IMG_EXTS])
lbl_count  = len(list(in_lbl_dir.glob("*.txt")))

print("───────────────────────────────────────────────────────────────")
print(f"  Input images  : {len(img_files)}")
print(f"  Input labels  : {lbl_count}")
if len(img_files) != lbl_count:
    print(f"  WARNING: image/label count mismatch — check your dataset")
print(f"  Multiplier    : {MULTIPLIER}x")
print(f"  Expected total: ~{len(img_files) * (MULTIPLIER + 1)}")
print(f"  Output        : {OUTPUT_DIR}/train")
print("───────────────────────────────────────────────────────────────\n")

# ── MAIN ─────────────────────────────────────────────────────────────────────
copied    = 0
augmented = 0
skipped   = 0
warnings  = 0

pbar = tqdm(img_files, desc="Augmenting", unit="img", dynamic_ncols=True)

for img_path in pbar:
    pbar.set_postfix({
        "copied": copied,
        "aug": augmented,
        "skip": skipped,
        "warn": warnings,
    })

    img = cv2.imread(str(img_path))
    if img is None:
        tqdm.write(f"  [SKIP] Cannot read {img_path.name}")
        skipped += 1
        continue

    img_rgb    = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_h, img_w = img_rgb.shape[:2]

    lbl_path = in_lbl_dir / (img_path.stem + ".txt")
    entries  = read_yolo_label(lbl_path)
    seg_mode = is_segmentation(entries)

    # Copy original as-is
    cv2.imwrite(str(out_img_dir / img_path.name), img)
    if lbl_path.exists():
        out_lbl_dir.joinpath(lbl_path.name).write_text(lbl_path.read_text())
    copied += 1

    bboxes, class_labels = seg_to_bboxes(entries) if entries else ([], [])

    if seg_mode and entries:
        instance_mask, class_map = polygons_to_mask(entries, img_w, img_h)
    else:
        instance_mask = np.zeros((img_h, img_w), dtype=np.uint8)
        class_map = {}

    for n in range(MULTIPLIER):
        try:
            result = transform(
                image=img_rgb,
                mask=instance_mask,
                bboxes=bboxes,
                class_labels=class_labels,
            )
        except Exception as e:
            tqdm.write(f"  [WARN] {img_path.name} copy {n}: {e}")
            warnings += 1
            continue

        aug_img  = cv2.cvtColor(result["image"], cv2.COLOR_RGB2BGR)
        aug_mask = result["mask"]
        aug_h, aug_w = aug_img.shape[:2]

        if seg_mode and class_map:
            new_entries = mask_to_polygons(
                aug_mask, class_map, aug_w, aug_h, min_area_px=MIN_AREA_PX
            )
        elif result["bboxes"]:
            new_entries = [
                (int(cls), list(bb))
                for cls, bb in zip(result["class_labels"], result["bboxes"])
            ]
        else:
            new_entries = []

        stem    = f"{img_path.stem}_aug{n:03d}"
        out_img = out_img_dir / f"{stem}{img_path.suffix}"
        out_lbl = out_lbl_dir / f"{stem}.txt"

        cv2.imwrite(str(out_img), aug_img)
        write_yolo_label(out_lbl, new_entries)
        augmented += 1

pbar.close()

print("\n───────────────────────────────────────────────────────────────") 
print(f"  Original copied : {copied}")
print(f"  Augmented added : {augmented}")
print(f"  Skipped         : {skipped}")
print(f"  Warnings        : {warnings}")
print(f"  Total images    : {copied + augmented}")
print(f"  Output          -> {OUTPUT_DIR}/train")
print("───────────────────────────────────────────────────────────────")

c:\Users\harry\Desktop\hsq-autobin\.venv\Lib\site-packages\albumentations\__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


───────────────────────────────────────────────────────────────
  Input images  : 1418
  Input labels  : 1418
  Multiplier    : 8x
  Expected total: ~12762
  Output        : ../data/augmented_tin/train
───────────────────────────────────────────────────────────────



Augmenting: 100%|██████████| 1418/1418 [36:14<00:00,  1.53s/img, copied=1417, aug=11336, skip=0, warn=0]


───────────────────────────────────────────────────────────────
  Original copied : 1418
  Augmented added : 11344
  Skipped         : 0
  Warnings        : 0
  Total images    : 12762
  Output          -> ../data/augmented_tin/train
───────────────────────────────────────────────────────────────
